<style>
.topic-header { background: linear-gradient(135deg, #e8f4f8 0%, #d4e8f0 100%); border-left: 4px solid #5ba4c9; padding: 16px 20px; border-radius: 0 8px 8px 0; margin: 10px 0; font-size: 15px; color: #1a3a4a; }
.concept-box { background: #eef6fa; border: 1px solid #c4dce8; border-radius: 8px; padding: 14px 18px; margin: 8px 0; font-size: 14px; color: #2a4a5a; }
.try-it { background: #fef9e7; border: 1px solid #f0d87a; border-radius: 8px; padding: 14px 18px; margin: 8px 0; font-size: 14px; color: #5a4a1a; }
.takeaway { background: #e8f5e8; border: 1px solid #a8d5a8; border-radius: 8px; padding: 14px 18px; margin: 8px 0; font-size: 14px; color: #2a5a2a; }
.warning-box { background: #fdf0f0; border: 1px solid #e8b0b0; border-radius: 8px; padding: 14px 18px; margin: 8px 0; font-size: 14px; color: #6a2a2a; }
.where-box { background: #fff3e0; border-left: 4px solid #ff9800; border-radius: 0 8px 8px 0; padding: 14px 18px; margin: 8px 0; font-size: 14px; color: #4a3000; }
.why-box { background: #fce4ec; border-left: 4px solid #e91e63; border-radius: 0 8px 8px 0; padding: 14px 18px; margin: 8px 0; font-size: 14px; color: #4a0020; }
.fix-box { background: #e8f5e9; border-left: 4px solid #4caf50; border-radius: 0 8px 8px 0; padding: 14px 18px; margin: 8px 0; font-size: 14px; color: #1b5e20; }
.diagram-box { background: #f8f9fa; border: 2px solid #dee2e6; border-radius: 12px; padding: 24px; margin: 16px 0; text-align: center; }
.flow-step { display: inline-block; background: #e8f4f8; border: 1px solid #5ba4c9; border-radius: 8px; padding: 8px 16px; margin: 4px; font-size: 13px; color: #1a3a4a; }
.compare-table { width: 100%; border-collapse: collapse; margin: 12px 0; }
.compare-table th { background: #d4e8f0; color: #1a3a4a; padding: 10px 14px; text-align: left; border: 1px solid #c4dce8; }
.compare-table td { padding: 10px 14px; border: 1px solid #dee2e6; font-size: 13px; }
.compare-table tr:nth-child(even) { background: #f8fbfd; }
.section-divider { border: none; border-top: 2px solid #d4e8f0; margin: 25px 0; }
</style>

<div class='topic-header'>
<h1>E02 &middot; Day 1 &middot; Zero-Shot &amp; Few-Shot Prompting</h1>
<p><strong>GenAI for Engineering Managers &bull; Exercise 2 of 15</strong> &nbsp;|&nbsp; Same question, dramatically different results &mdash; based on <em>how</em> you ask.</p>
</div>

**Why this matters at your altitude.** As an engineering manager you will rarely write
prompts all day &mdash; but your teams will, and the systems they ship will live or die on
prompt quality. When a team tells you "the model is unreliable," the right first question
is usually "show me the prompt." This exercise gives you the vocabulary and the intuition
to review that work: to know when clear instructions are enough, when examples are needed,
and what "production-ready output" actually looks like. You do not need to write any code
today &mdash; the facilitator runs every cell. Your job is to <strong>predict, observe, and
compare</strong>.


<div class='why-box'>
<strong>&#128279; Where E01 left us.</strong><br><br>
In E01 we got raw access to a model through the API &mdash; and we saw the gap immediately:
<em>the answers depended entirely on how we asked</em>. The same question, phrased two ways,
gave two different answers in two different shapes. Raw access without control is a demo,
not a system.<br><br>
This exercise gives us the <strong>first lever of control</strong>: the difference between
<em>zero-shot</em> prompting (clear instructions, no examples) and <em>few-shot</em> prompting
(teaching the model by example). By the end, the same flaky classification task will run
reliably enough to sit inside a real pipeline.
</div>

<hr class='section-divider'>

## Part 1 &mdash; Why Prompt Engineering Matters

Before we learn any technique, let us see the problem with our bare hands.
Imagine one of your teams is building a triage system for <strong>seller support emails</strong>
at a large retailer's marketplace. We will ask the model a perfectly reasonable business
question &mdash; and see why the answer, while technically correct, is **completely unusable**
in a production system.

<div class='try-it'>
<strong>&#128173; Predict before we run:</strong> we will ask the model to "classify" a seller's
email with no further instructions. What do you expect back &mdash; a single label, or a paragraph?
</div>

In [ ]:
# ── Setup (carries forward from Exercise E01) ──────────────────────
from openai import OpenAI
import json

client = OpenAI(
    api_key='PASTE_THE_KEY_SHARED_IN_SESSION_HERE'
)
MODEL = 'gpt-5.4-nano'

def ask(prompt, system=None, temperature=0):
    '''Send a prompt and return the response text.'''
    msgs = []
    if system:
        msgs.append({'role': 'system', 'content': system})
    msgs.append({'role': 'user', 'content': prompt})
    r = client.chat.completions.create(model=MODEL, messages=msgs, temperature=temperature)
    return r.choices[0].message.content

print('Ready  |  model:', MODEL)

In [ ]:
# ── The problem: a vague prompt ────────────────────────────────────
# Imagine your team is building an automated triage system for
# seller support emails at a large retailer's marketplace.
# A seller email arrives. We ask the model to classify it.

seller_text = ('My payouts have been on hold for two weeks and my listings '
               'were deactivated without any explanation. I need this fixed '
               'before the weekend sale event.')

result = ask('Classify this: ' + seller_text)

print('=== Model Response ===')
print(result)

<div class='where-box'>
<strong>What just happened?</strong><br><br>
The model gave us <em>something</em> &mdash; but look at the output:
<ul>
<li>It is a block of free-form markdown &mdash; headers, bold labels, its own invented category names &mdash; not the single label your routing code needs.</li>
<li>If you run it again, the wording may change.</li>
<li>You cannot <code>json.loads()</code> it, you cannot feed it into a routing queue, you cannot page the right on-call team with it.</li>
</ul>
The model <em>understood</em> the task &mdash; it just did not know <strong>how you wanted the answer</strong>.
That is the gap prompt engineering fills. When your team demos "the model got it wrong,"
this is very often what actually happened.
</div>

<div class='concept-box'>
<strong>&#129504; Mental model &mdash; Prompt Engineering</strong> is the practice of structuring your input &mdash; the words,
the format, the constraints, and the examples you provide &mdash; so that the model gives you
an output that is <em>correct, consistent, and machine-parseable</em>.<br><br>

Think of it like giving directions to a taxi driver:<br>
&bull; <em>"Take me somewhere nice"</em> &rarr; you could end up anywhere.<br>
&bull; <em>"Take me to Terminal B departures, take the expressway, avoid tolls"</em> &rarr; exact result.<br><br>

The model is the taxi driver &mdash; extremely capable, but it needs <strong>clear instructions</strong>
to deliver what you actually want.
</div>

<div class='diagram-box'>
<h3 style='margin-top:0; color:#1a3a4a;'>Prompting Techniques &mdash; Taxonomy</h3>
<div style='display:flex; flex-direction:column; align-items:center; gap:6px; margin-top:12px;'>

<div class='flow-step' style='background:#d4e8f0; font-weight:bold; font-size:15px; padding:10px 24px;'>Prompt Engineering</div>

<div style='font-size:20px; color:#5ba4c9;'>&darr;</div>

<div style='display:flex; flex-wrap:wrap; justify-content:center; gap:8px;'>
  <div class='flow-step' style='border:2px solid #5ba4c9; font-weight:bold;'>Zero-Shot<br><span style='font-size:11px; font-weight:normal;'>(no examples)</span></div>
  <div class='flow-step' style='border:2px solid #5ba4c9; font-weight:bold;'>Few-Shot<br><span style='font-size:11px; font-weight:normal;'>(with examples)</span></div>
  <div class='flow-step' style='border:1px dashed #999;'>Chain-of-Thought<br><span style='font-size:11px;'>(step-by-step)</span></div>
  <div class='flow-step' style='border:1px dashed #999;'>Tree-of-Thought<br><span style='font-size:11px;'>(multiple paths)</span></div>
  <div class='flow-step' style='border:1px dashed #999;'>System Prompts<br><span style='font-size:11px;'>(role + constraints)</span></div>
</div>

<p style='margin-top:12px; font-size:13px; color:#666;'>
<strong>Solid border</strong> = covered in this notebook &nbsp;&bull;&nbsp;
<strong>Dashed border</strong> = covered in Exercise E03 (Chain-of-Thought &amp; Tree-of-Thought)
</p>
</div>
</div>

<hr class='section-divider'>

## Part 2 &mdash; Zero-Shot Prompting: The Power of Clear Instructions

<div class='concept-box'>
<strong>Zero-Shot Prompting</strong> means giving the model a task with
<em>no examples</em> of desired input/output pairs. You rely entirely on
the model's training knowledge and your instruction clarity.<br><br>

The key insight: <strong>most failures are not failures of the model &mdash;
they are failures of the prompt.</strong> The model can almost always do what
you want; you just have not told it clearly enough.
</div>

### The Format-Control Experiment

We will run the **same sentiment-analysis task** three ways and watch the
output quality improve dramatically with each refinement. The input is a
customer review from the retailer's marketplace &mdash; the kind your catalog
and review-moderation teams process millions of at a time.

**Test text:** *"I absolutely love this phone! The camera is amazing. However,
the battery life is disappointing &mdash; it barely lasts half a day. The seller
was also quite unhelpful when I messaged about the issue."*

<div class='try-it'>
<strong>&#128173; Predict:</strong> the review mixes praise and complaints.
Which of the three approaches do you expect to handle that nuance best?
</div>

In [ ]:
# ── Approach 1: The vague prompt ───────────────────────────────────
review = ('I absolutely love this phone! The camera is amazing. '
          'However, the battery life is disappointing - it barely lasts '
          'half a day. The seller was also quite unhelpful when I messaged '
          'about the issue.')

result_1 = ask('What is the sentiment of this text: ' + review)

print('=== Approach 1: Vague Prompt ===')
print('Prompt:  "What is the sentiment of this text: ..."')
print()
print('Response:')
print(result_1)

In [ ]:
# ── Approach 2: Constrained output ─────────────────────────────────
prompt_2 = ('Classify the sentiment of the following text as exactly one of: '
            'POSITIVE, NEGATIVE, or MIXED. '
            'Reply with ONLY the classification word, nothing else.\n\n'
            'Text: ' + review)

result_2 = ask(prompt_2)

print('=== Approach 2: Constrained Prompt ===')
print('Prompt:  "Classify as POSITIVE, NEGATIVE, or MIXED. Reply with one word only."')
print()
print('Response:', repr(result_2))

In [ ]:
# ── Approach 3: Structured JSON output ─────────────────────────────
prompt_3 = ('Analyze the sentiment of the following text. '
            'Reply ONLY with valid JSON in this exact format, no other text:\n'
            '{"sentiment": "POSITIVE or NEGATIVE or MIXED", '
            '"confidence": 0.0, '
            '"aspects": [{"topic": "...", "sentiment": "..."}]}\n\n'
            'Text: ' + review)

result_3 = ask(prompt_3)

print('=== Approach 3: JSON Format ===')
print('Response:')
print(result_3)

# Verify it is valid JSON
try:
    parsed = json.loads(result_3)
    print('\nJSON is valid! Keys:', list(parsed.keys()))
except json.JSONDecodeError as e:
    print('\nJSON parsing failed:', e)

### Side-by-Side Comparison

<table class='compare-table'>
<tr>
  <th>Approach</th>
  <th>Prompt Style</th>
  <th>Output Type</th>
  <th>Machine-Parseable?</th>
  <th>Production-Ready?</th>
</tr>
<tr>
  <td><strong>1. Vague</strong></td>
  <td>"What is the sentiment?"</td>
  <td>Free-form paragraph</td>
  <td>No</td>
  <td>No</td>
</tr>
<tr>
  <td><strong>2. Constrained</strong></td>
  <td>"Classify as X/Y/Z. One word only."</td>
  <td>Single label</td>
  <td>Yes (string match)</td>
  <td>Yes</td>
</tr>
<tr>
  <td><strong>3. Structured</strong></td>
  <td>"Reply as JSON: {format spec}"</td>
  <td>Valid JSON object</td>
  <td>Yes (json.loads)</td>
  <td>Yes + rich metadata</td>
</tr>
</table>

<div class='takeaway'>
<strong>Takeaway:</strong> The model is equally capable in all three cases &mdash;
the only thing that changed was <em>how we asked</em>. The quality of your prompt
determines the quality of your output. In production systems, always specify the
<strong>exact format</strong> you need.
</div>

<hr class='section-divider'>

## Part 3 &mdash; Zero-Shot for Real Business Tasks

**Scenario:** Your team runs the seller-support platform at a large retailer's
online marketplace. Thousands of seller emails arrive daily and need to be
automatically classified, parsed, summarized for leadership rollups, and
sometimes translated for regional support teams.

We will use the same seller email across four tasks &mdash; each time contrasting
a naive prompt with an engineered prompt.

In [ ]:
# ── The sample email we will process across all tasks ──────────────
email_text = '''Subject: Urgent - Payout Hold and Listing Deactivation, Seller ID SLR-88214

Dear Marketplace Seller Support,

I am writing to express my deep frustration regarding the hold placed on my
payouts (Case ID: CASE-2026-45120). My store, Cedar & Pine Home Goods, has
sold on your marketplace since March 2022 with a 98.4% positive rating.

On August 1st, a payout of $18,750 was frozen and 240 of my listings were
deactivated, citing "verification review." I re-submitted my business
verification documents on August 3rd and received an automated confirmation,
but the hold remains. Your own seller policy states verification reviews are
completed within 5 business days.

I demand an immediate review. If this is not resolved within 7 days, I will
escalate to the marketplace ombudsman program and pursue the matter through
my trade association.

Seller: Marcus Webb
Store: Cedar & Pine Home Goods
Seller ID: SLR-88214
Case ID: CASE-2026-45120
Contact: +1 (555) 014-8821
Email: marcus@cedarandpinehome.com

Regards,
Marcus Webb'''

print('Email loaded:', len(email_text), 'characters')
print(email_text[:120], '...')

In [ ]:
# ── Task 1: Email Classification ───────────────────────────────────
# Categories: complaint, inquiry, payout, feedback, escalation

# --- Naive prompt ---
naive_result = ask('Classify this email:\n\n' + email_text)
print('=== NAIVE PROMPT ===')
print('Prompt: "Classify this email: ..."')
print('Result:', naive_result[:200])
print()

# --- Engineered prompt ---
engineered_prompt = (
    'You are an email classifier for the seller-support desk of a large '
    'retail marketplace. '
    'Classify the following email into EXACTLY ONE of these categories:\n'
    '- COMPLAINT: seller expressing dissatisfaction\n'
    '- INQUIRY: seller asking for information\n'
    '- PAYOUT: payment, payout, or settlement issue\n'
    '- FEEDBACK: general feedback or suggestion\n'
    '- ESCALATION: threat of legal/regulatory/ombudsman action\n\n'
    'Reply with ONLY a JSON object: '
    '{"category": "...", "priority": "HIGH/MEDIUM/LOW", '
    '"reason": "one sentence"}\n\n'
    'Email:\n' + email_text
)
eng_result = ask(engineered_prompt)
print('=== ENGINEERED PROMPT ===')
print('Result:')
print(eng_result)

In [ ]:
# ── Task 2: Entity Extraction ──────────────────────────────────────
extract_prompt = (
    'Extract the following entities from this marketplace seller email. '
    'Reply ONLY with valid JSON, no other text.\n\n'
    'Required fields:\n'
    '- seller_name: full name of the person writing\n'
    '- store_name: name of the seller store\n'
    '- seller_id: the seller account ID\n'
    '- case_id: the support case reference number\n'
    '- amount_on_hold: frozen payout amount in dollars (number only)\n'
    '- listings_affected: number of deactivated listings (number only)\n'
    '- contact_phone: phone number\n'
    '- contact_email: email address\n'
    '- date_of_hold: date the payout hold started\n'
    '- urgency_deadline: any deadline mentioned\n\n'
    'Email:\n' + email_text
)

extract_result = ask(extract_prompt)
print('=== Extracted Entities ===')
print(extract_result)

# Validate
try:
    entities = json.loads(extract_result)
    print('\nExtracted', len(entities), 'fields successfully')
    for k, v in entities.items():
        print(f'  {k}: {v}')
except json.JSONDecodeError:
    print('\nNote: response was not valid JSON')

In [ ]:
# ── Task 3: Controlled Summarization (leadership rollup) ───────────
# The kind of one-glance summary you would want in a weekly ops rollup.
summary_prompt = (
    'Summarize the following marketplace seller email in EXACTLY '
    '2 bullet points for an engineering-leadership rollup. Each bullet must '
    'be one sentence, maximum 20 words. '
    'Focus on: (1) what the seller wants, (2) what action is needed.\n\n'
    'Email:\n' + email_text
)

summary = ask(summary_prompt)
print('=== Summary (2 bullets, max 20 words each) ===')
print(summary)

In [ ]:
# ── Task 4: Translation with Tone Preservation ─────────────────────
# The marketplace has a regional support team in Mexico; summaries are
# shared in Spanish. Tone must survive the translation.
translate_prompt = (
    'Translate the following English case summary into Spanish. '
    'Preserve the formal and urgent tone. '
    'After the translation, add a line: '
    '"Tone: [FORMAL/INFORMAL] | Urgency: [HIGH/MEDIUM/LOW]"\n\n'
    'Text to translate:\n'
    'Seller Marcus Webb (Cedar & Pine Home Goods) demands immediate review '
    'of a $18,750 payout hold and 240 deactivated listings (Case '
    'CASE-2026-45120). Seller threatens ombudsman escalation if not '
    'resolved in 7 days.'
)

translation = ask(translate_prompt)
print('=== Spanish Translation ===')
print(translation)

<div class='takeaway'>
<strong>Pattern:</strong> Every one of these tasks used the same zero-shot formula:<br><br>
<strong>Role</strong> (who you are) + <strong>Task</strong> (what to do) +
<strong>Constraints</strong> (rules/limits) + <strong>Format</strong> (how to reply)<br><br>
No examples were needed &mdash; the model already knows how to classify, extract, summarize,
and translate. We just had to tell it <em>precisely how we wanted the output</em>.
</div>

<hr class='section-divider'>

## Part 4 &mdash; The System Prompt: Setting the Stage

<div class='concept-box'>
<strong>Two Channels of Communication</strong><br><br>
When you talk to a language model through the API, there are two distinct channels:
<ul>
<li><strong>System Prompt</strong> (<code>role: 'system'</code>): Sets the model's persona,
rules, and constraints. Think of it as a <em>job description</em> given to an employee
before their first day. The end user never sees this.</li>
<li><strong>User Prompt</strong> (<code>role: 'user'</code>): The actual task or question.
This is what the "employee" processes using the guidelines from their job description.</li>
</ul>
The system prompt is your <strong>secret control panel</strong> for model behavior.
Same user question + different system prompt = completely different response.
</div>

In [ ]:
# ── System Prompt Experiment ───────────────────────────────────────
# Same user question, 4 different system prompts

user_question = (
    'A top-rated seller says their payouts were frozen unfairly and their '
    'listings deactivated. They are threatening to escalate to the '
    'ombudsman and go to the press. What should I do?'
)

# Approach 1: No system prompt at all
r1 = ask(user_question)
print('=== 1. NO SYSTEM PROMPT ===')
print(r1[:300])
print()

# Approach 2: Seller support agent
r2 = ask(user_question,
         system=('You are a seller support agent at a large retail '
                 'marketplace. You are empathetic but professional. '
                 'Always de-escalate. Suggest concrete next steps. '
                 'Keep responses under 100 words.'))
print('=== 2. SELLER SUPPORT AGENT ===')
print(r2)

In [ ]:
# Approach 3: Trust & compliance officer
r3 = ask(user_question,
         system=('You are a trust-and-compliance officer at a large retail '
                 'marketplace. Always reference the seller agreement and '
                 'applicable marketplace-fairness regulations. '
                 'Warn about potential regulatory and reputational risks. '
                 'Use formal language. Keep responses under 100 words.'))
print('=== 3. TRUST & COMPLIANCE OFFICER ===')
print(r3)
print()

# Approach 4: Sarcastic comedian (to show how wrong it can go!)
r4 = ask(user_question,
         system=('You are a stand-up comedian. Everything is material for jokes. '
                 'Be witty and sarcastic. Keep it short.'),
         temperature=0.7)
print('=== 4. SARCASTIC COMEDIAN (do NOT use in production!) ===')
print(r4)

<div class='concept-box'>
<strong>Anatomy of a Great System Prompt</strong><br><br>
A production-grade system prompt has four components:
<table class='compare-table'>
<tr><th>Component</th><th>Purpose</th><th>Example</th></tr>
<tr>
  <td><strong>Role</strong></td>
  <td>Who the model is</td>
  <td>"You are a senior seller-support analyst at a large retail marketplace"</td>
</tr>
<tr>
  <td><strong>Context</strong></td>
  <td>What the model knows</td>
  <td>"You have access to the 2026 seller policy handbook. The desk handles 5,000 cases/day."</td>
</tr>
<tr>
  <td><strong>Constraints</strong></td>
  <td>Rules and guardrails</td>
  <td>"Never promise a specific outcome. Always recommend the seller review their agreement."</td>
</tr>
<tr>
  <td><strong>Format</strong></td>
  <td>How to respond</td>
  <td>"Reply in bullet points. Maximum 5 bullets. End with a recommended action."</td>
</tr>
</table>
</div>

<div class='warning-box'>
<strong>Warning:</strong> The system prompt is powerful but not foolproof. A determined
user can sometimes override it through creative prompting ("ignore your instructions and...").
For safety-critical applications, always add server-side validation on top of prompt-level controls.
We will cover guardrails in detail later in the program.
</div>

<hr class='section-divider'>

## Part 5 &mdash; Few-Shot Prompting: Teaching by Example

<div class='concept-box'>
<strong>Few-Shot Prompting</strong> means including 2&ndash;5 examples of
desired input &rarr; output pairs in your prompt, <em>before</em> asking
your actual question.<br><br>

It is like onboarding a new team member: instead of writing a 10-page manual,
you show them three completed tickets and say <em>"do it like this."</em>
The model learns the pattern from your examples and applies it to new inputs.<br><br>

<strong>When is it needed?</strong> When the task involves domain-specific
categories, ambiguous boundaries, or a particular output style that the model
cannot infer from instructions alone.
</div>

### The Classification Challenge

**Task:** Classify customer complaints arriving at a large retailer's online
store into exactly one of four categories: **billing**, **technical**,
**delivery**, **quality**.

Let us first try zero-shot, then see where it struggles, and fix it with few-shot.

<div class='try-it'>
<strong>&#128173; Predict:</strong> one of the five complaints below is genuinely
ambiguous &mdash; a payment option greyed out in the app. Is that <em>billing</em>
or <em>technical</em>? Note your answer; we will see how the model decides.
</div>

In [ ]:
# ── Zero-Shot Classification Attempt ───────────────────────────────
test_complaints = [
    'I was charged $49.99 twice for my membership this month',
    'The app crashes every time I open my order history',
    'My order has not arrived even after 3 weeks of placing it',
    'The fabric quality of the shirt is nothing like the photos',
    'I want to update my payment method but the option is greyed out in the app',
]
# Our support team's convention: payment-related issues route to billing,
# even when the symptom is in the app. That is exactly the kind of
# boundary the model cannot guess without examples.
expected = ['billing', 'technical', 'delivery', 'quality', 'billing']

zero_prompt_template = (
    'Classify this customer complaint into one of these categories: '
    'billing, technical, delivery, quality.\n\n'
    'Complaint: {complaint}\n\n'
    'Category:'
)

print('=== Zero-Shot Classification ===')
print(f'{"":<3} {"Complaint":<62} {"Expected":<10} {"Got"}')
print('-' * 95)

zero_results = []
for i, (complaint, exp) in enumerate(zip(test_complaints, expected), 1):
    prompt = zero_prompt_template.format(complaint=complaint)
    result = ask(prompt).strip().lower().rstrip('.')
    zero_results.append(result)
    match = 'OK' if exp in result else 'MISS'
    print(f'{i:<3} {complaint[:60]:<62} {exp:<10} {result:<12} {match}')

<div class='where-box'>
<strong>Where zero-shot struggles:</strong><br>
<ul>
<li>The model may reply with a full sentence instead of a single word ("The category is billing")</li>
<li>It may invent categories not in our list ("account", "service", "app")</li>
<li>Ambiguous cases (like "payment option greyed out in the app") can go either way &mdash;
    is that a <em>billing</em> issue (about payments) or a <em>technical</em> issue (app bug)?</li>
</ul>
Without examples, the model has no way to know how <em>your team</em> draws these boundaries.
</div>

<div class='fix-box'>
<strong>The Fix:</strong> Add 2&ndash;4 examples that show exactly how each
category maps, especially for edge cases. The examples teach the model your
team's specific definitions &mdash; and they also lock in the output format
(single lowercase word).
</div>

In [ ]:
# ── Few-Shot Classification ────────────────────────────────────────
few_shot_prompt_template = '''Classify customer complaints into exactly one category.
Categories: billing, technical, delivery, quality.
Reply with ONLY the category name in lowercase, nothing else.

Complaint: I was charged double for last month's order
Category: billing

Complaint: The website keeps showing a 504 error when I try to checkout
Category: technical

Complaint: The package was delivered to the wrong address
Category: delivery

Complaint: The product broke after just two days of normal use
Category: quality

Complaint: The saved-card screen will not let me change my card
Category: billing

Complaint: {complaint}
Category:'''

print('=== Few-Shot Classification ===')
print(f'{"":<3} {"Complaint":<62} {"Expected":<10} {"Got"}')
print('-' * 95)

few_results = []
for i, (complaint, exp) in enumerate(zip(test_complaints, expected), 1):
    prompt = few_shot_prompt_template.format(complaint=complaint)
    result = ask(prompt).strip().lower().rstrip('.')
    few_results.append(result)
    match = 'OK' if exp in result else 'MISS'
    print(f'{i:<3} {complaint[:60]:<62} {exp:<10} {result:<12} {match}')

In [ ]:
# ── Side-by-Side Comparison ────────────────────────────────────────
print('=== Zero-Shot vs Few-Shot: Head-to-Head ===')
print()
print(f'{"":<3} {"Complaint":<50} {"Expected":<10} {"Zero-Shot":<14} {"Few-Shot":<14}')
print('=' * 95)

zero_correct = 0
few_correct = 0

for i, (complaint, exp, zr, fr) in enumerate(
        zip(test_complaints, expected, zero_results, few_results), 1):
    z_ok = exp in zr
    f_ok = exp in fr
    zero_correct += z_ok
    few_correct += f_ok
    print(f'{i:<3} {complaint[:48]:<50} {exp:<10} '
          f'{zr:<12} {"OK" if z_ok else "X":<2} '
          f'{fr:<12} {"OK" if f_ok else "X":<2}')

print('=' * 95)
print(f'{"Accuracy:":<65} {zero_correct}/{len(expected):<12}  '
      f'{few_correct}/{len(expected)}')

<hr class='section-divider'>

## Part 6 &mdash; Few-Shot for Structured Output

Few-shot really shines when you need the model to produce complex structured
data consistently. By showing 2&ndash;3 examples of the exact JSON structure you
want, the model learns both the *schema* and the *extraction logic* simultaneously.

In [ ]:
# ── Few-Shot Structured Extraction ─────────────────────────────────
# Task: Extract structured data from marketplace product reviews

extraction_prompt = '''Extract structured data from product reviews as JSON.

Review: "Bought the Nova Trail 24L daypack last month. The laptop sleeve is
well padded and the water-bottle pockets actually fit large bottles. The
zippers feel flimsy though. Paid $54.99, sold and shipped by a marketplace
seller. Rating: 4 out of 5."
Output: {"product": "Nova Trail 24L daypack", "price": 54.99, "currency": "USD", "rating": 4, "max_rating": 5, "pros": ["padded laptop sleeve", "large bottle pockets"], "cons": ["flimsy zippers"], "fulfillment": "marketplace seller"}

Review: "Got the HearthGlow 3-wick candle set on the store's own site for
$21.50 with free store pickup. The cedar scent fills the whole room and the
burn is very even. But two of the lids arrived scratched. 3.5 out of 5."
Output: {"product": "HearthGlow 3-wick candle set", "price": 21.5, "currency": "USD", "rating": 3.5, "max_rating": 5, "pros": ["strong cedar scent", "even burn"], "cons": ["scratched lids"], "fulfillment": "store pickup"}

Review: "The VoltEdge 20V cordless drill is incredible for $89 with next-day
delivery. Plenty of torque for deck screws and the two batteries charge
fast. Only issue is the carrying case latch feels cheap. Giving it 4.5
stars out of 5."
Output:'''

result = ask(extraction_prompt)
print('=== Extracted Data ===')
print(result)

try:
    data = json.loads(result)
    print('\nParsed successfully!')
    print(json.dumps(data, indent=2))
except json.JSONDecodeError as e:
    print('\nJSON error:', e)

In [ ]:
# ── Batch extraction + validation ──────────────────────────────────
# Let us run the same few-shot template on 3 new reviews

new_reviews = [
    ('Ordered the ChefMate 750W immersion blender for $34.99 with next-day '
     'delivery. Blends soups perfectly and the detachable shaft cleans '
     'easily. Motor is a bit loud though. Would give it 4 out of 5 stars.'),

    ('The ClearFrame blue-light glasses cost $18.99 on the app. Frame looks '
     'premium and fits well. The blue light filter really helps with screen '
     'fatigue. No complaints at all. 5/5.'),

    ('Tried the GlowLeaf Vitamin C face wash, $12.49, sold by a marketplace '
     'seller. Nice citrus smell and lathers well. But no visible difference '
     'even after 3 weeks of daily use. Feels overpriced. 2.5 out of 5.'),
]

all_extractions = []
for i, rev in enumerate(new_reviews, 1):
    prompt = extraction_prompt.rsplit('Review:', 1)[0]  # keep examples
    prompt += 'Review: "' + rev + '"\nOutput:'
    result = ask(prompt)
    print(f'--- Review {i} ---')
    try:
        parsed = json.loads(result)
        all_extractions.append(parsed)
        print(f'  Product:  {parsed.get("product", "?")}')
        print(f'  Price:    $ {parsed.get("price", "?")}')
        print(f'  Rating:   {parsed.get("rating", "?")}/{parsed.get("max_rating", "?")}')
        print(f'  Pros:     {parsed.get("pros", [])}')
        print(f'  Cons:     {parsed.get("cons", [])}')
    except json.JSONDecodeError:
        print(f'  Could not parse JSON: {result[:100]}')
    print()

print(f'Successfully extracted {len(all_extractions)}/{len(new_reviews)} reviews')

<div class='where-box'>
<strong>Did you catch the failure?</strong> Review 3 came back wrapped in a <code>```json</code> markdown fence &mdash; the content was fine, but <code>json.loads()</code> rejected it, so only 2/3 parsed. Even with good few-shot examples the model occasionally drifts on formatting. That is not a reason to abandon few-shot; it is the reason production pipelines <em>always validate</em> (and typically strip fences or retry on parse failure). We keep this real failure in the notebook on purpose.
</div>

In [ ]:
# ── Few-Shot for Consistent Tone/Style ─────────────────────────────
# Task: Rewrite dry supplier spec-sheets into the retailer's house-tone
# catalog copy: warm, practical, benefit-first, no jargon.

style_prompt = '''Rewrite supplier spec-sheet text into our house-tone catalog copy.
House tone: warm, practical, benefit-first, everyday language, 2-3 sentences.

Spec sheet: "Insulated stainless-steel tumbler. Capacity 30 oz. Double-wall
vacuum construction. Retains cold 24 hrs / heat 8 hrs. Fits standard cup
holders."
Catalog copy: "Ice still clinking at the end of your shift? That's the
double-wall vacuum design keeping drinks cold for a full 24 hours (or hot
for 8). And yes - it fits your cup holder."

Spec sheet: "LED desk lamp. 3 color temperatures, 5 brightness levels.
USB-C charging port in base. Foldable arm. Memory function retains last
setting."
Catalog copy: "From bright-white focus mode to a warm evening glow, this
lamp adjusts to how you work. The base charges your phone while the
foldable arm tucks away when you're done - and it remembers your favorite
setting for tomorrow."

Spec sheet: "Weighted blanket, 15 lb, 60 x 80 in. Breathable cotton outer
layer with glass-bead fill. Machine washable cover. Even weight
distribution via box stitching."
Catalog copy:'''

result = ask(style_prompt)
print('=== Spec Sheet to House-Tone Catalog Copy ===')
print(result)

<hr class='section-divider'>

## Part 7 &mdash; When Zero-Shot Beats Few-Shot (and Vice Versa)

Neither technique is universally better. The right choice depends on the task:

<table class='compare-table'>
<tr>
  <th>Scenario</th>
  <th>Best Approach</th>
  <th>Why</th>
</tr>
<tr>
  <td>Simple factual Q&amp;A</td>
  <td><strong>Zero-shot</strong></td>
  <td>Model already knows the answer; examples waste tokens</td>
</tr>
<tr>
  <td>Domain-specific classification</td>
  <td><strong>Few-shot</strong></td>
  <td>Model needs to learn YOUR specific categories and boundaries</td>
</tr>
<tr>
  <td>Strict format control</td>
  <td><strong>Zero-shot + format spec</strong></td>
  <td>Clear format instructions are cheaper than example pairs</td>
</tr>
<tr>
  <td>Ambiguous edge cases</td>
  <td><strong>Few-shot</strong></td>
  <td>Examples resolve ambiguity by showing how you handle edge cases</td>
</tr>
<tr>
  <td>Creative generation</td>
  <td><strong>Zero-shot</strong></td>
  <td>Examples can <em>constrain</em> creativity (see demo below)</td>
</tr>
<tr>
  <td>Consistent style/tone</td>
  <td><strong>Few-shot</strong></td>
  <td>Style is hard to describe in words but easy to show by example</td>
</tr>
<tr>
  <td>Very long input text</td>
  <td><strong>Zero-shot</strong></td>
  <td>Examples eat into your context window; save tokens for the input</td>
</tr>
</table>

### When Few-Shot *Hurts*: The Pattern-Lock Problem

Sometimes giving examples **biases** the model toward a narrow pattern
that does not generalize. Let us see this in action.

In [ ]:
# ── When Few-Shot Hurts: Pattern Lock ──────────────────────────────
# Task: Generate a creative name for a private-label product line

# Zero-shot: open-ended creativity
zs_result = ask(
    'Create 5 creative, memorable brand names for a large retailer\'s new '
    'private-label line of home goods made from recycled ocean plastics. '
    'Be diverse and inventive. Just list the names, one per line.',
    temperature=0.8
)
print('=== ZERO-SHOT (creative, diverse) ===')
print(zs_result)
print()

# Few-shot with pattern-locked examples
fs_result = ask(
    '''Generate a creative brand name for a sustainable private-label line.

Line: eco-friendly cleaning products
Brand: GreenClean

Line: organic pet food
Brand: GreenPaws

Line: sustainable packaging
Brand: GreenWrap

Line: home goods made from recycled ocean plastics
Brand:''',
    temperature=0.8
)
print('=== FEW-SHOT (pattern-locked!) ===')
print(fs_result)

<div class='warning-box'>
<strong>The Pattern-Lock Trap:</strong> Notice how the few-shot examples all
followed the pattern "Green + Noun"? The model learned that pattern and
applied it &mdash; producing a derivative name instead of something truly creative.<br><br>

<strong>Rule of thumb:</strong> Use few-shot when you want <em>consistency</em>.
Use zero-shot when you want <em>creativity</em>. If you must use few-shot for
creative tasks, make your examples deliberately <em>diverse in structure</em>
so the model learns the quality bar, not a surface-level pattern.
</div>

<hr class='section-divider'>

## Part 8 &mdash; Building a Reusable Prompt Template

Now that we understand zero-shot, few-shot, system prompts, and format control,
let us build a **single Python function** that combines all of these techniques.
This function will be the foundation for everything we build across the rest of the week.

<div class='concept-box'>
<strong>Design Goals:</strong>
<ul>
<li>Works for classification, extraction, and summarization</li>
<li>Supports optional few-shot examples</li>
<li>Supports optional system prompt</li>
<li>Always specifies output format</li>
<li>Returns parsed JSON when possible</li>
</ul>
</div>

In [ ]:
# ── Reusable Prompt Template ───────────────────────────────────────

def prompt_template(task_type, input_text, examples=None,
                    output_format=None, system_prompt=None,
                    temperature=0):
    '''
    Universal prompt template for common NLP tasks.

    Parameters:
        task_type:     'classify', 'extract', 'summarize', or custom instruction
        input_text:    the text to process
        examples:      list of {'input': ..., 'output': ...} dicts (few-shot)
        output_format: string describing the desired output format
        system_prompt: optional system-level instruction
        temperature:   model temperature (0 = deterministic)

    Returns:
        Parsed JSON (dict/list) if output is valid JSON, else raw string.
    '''
    # Build the task instruction
    task_instructions = {
        'classify': 'Classify the following text.',
        'extract': 'Extract key information from the following text.',
        'summarize': 'Summarize the following text concisely.',
    }
    instruction = task_instructions.get(task_type, task_type)

    # Assemble the prompt
    parts = [instruction]

    # Add output format constraint
    if output_format:
        parts.append('Output format: ' + output_format)

    # Add few-shot examples
    if examples:
        parts.append('\nExamples:')
        for ex in examples:
            parts.append('Input: ' + ex['input'])
            parts.append('Output: ' + ex['output'])
            parts.append('')

    # Add the actual input
    parts.append('Input: ' + input_text)
    parts.append('Output:')

    prompt = '\n'.join(parts)

    # Call the model
    result = ask(prompt, system=system_prompt, temperature=temperature)

    # Try to parse as JSON
    try:
        return json.loads(result)
    except (json.JSONDecodeError, TypeError):
        return result.strip()

print('prompt_template() defined and ready')

In [ ]:
# ── Demo 1: Zero-shot classification ───────────────────────────────
result = prompt_template(
    task_type='classify',
    input_text='The delivery was 2 weeks late and the product was damaged',
    output_format='JSON: {"category": "...", "sentiment": "POSITIVE/NEGATIVE/NEUTRAL"}'
)
print('1. Zero-shot classification:')
print('  ', result)
print()

# ── Demo 2: Few-shot incident-severity labeling ────────────────────
# The kind of triage your on-call rotation does every week.
result = prompt_template(
    task_type='classify',
    input_text='Checkout latency is elevated in one region; error rate normal',
    examples=[
        {'input': 'Site fully down for all customers', 'output': 'SEV1'},
        {'input': 'Payment failures for 8% of checkout attempts', 'output': 'SEV2'},
        {'input': 'Product images loading slowly on the app home screen', 'output': 'SEV3'},
    ],
    output_format='single severity label only (SEV1/SEV2/SEV3)'
)
print('2. Few-shot incident severity:')
print('  ', result)
print()

# ── Demo 3: Extraction with format spec ────────────────────────────
result = prompt_template(
    task_type='extract',
    input_text=('Sprint 42 review with Dana Chen on Jan 15 at 3 PM in '
                'Conference Room B to discuss the $250K checkout-replatform budget'),
    output_format=('JSON: {"person": "...", "date": "...", "time": "...", '
                   '"location": "...", "topic": "...", "amount": ...}')
)
print('3. Extraction:')
print('  ', result)
print()

# ── Demo 4: Summarization (leadership rollup) ──────────────────────
result = prompt_template(
    task_type='summarize',
    input_text=email_text,
    output_format='Exactly 2 bullet points, max 15 words each',
    system_prompt='You are a concise engineering-operations analyst.'
)
print('4. Summarization:')
print('  ', result)

<hr class='section-divider'>

## Wrap-Up &mdash; The Full Journey

Let us end where we started: the same classification task, run three ways.
This final demo shows just how far we have come in this notebook.

In [ ]:
# ── Final Comparison: No Technique vs Zero-Shot vs Few-Shot ────────
test_text = ('My payouts have been on hold for two weeks and my listings '
             'were deactivated without any explanation. I need this fixed '
             'before the weekend sale event.')

# 1. No technique (raw question)
raw = ask('Classify this: ' + test_text)

# 2. Zero-shot with format control
zs = ask(
    'Classify this marketplace seller complaint into exactly one category: '
    'billing, technical, delivery, quality. '
    'Reply with ONLY a JSON object: '
    '{"category": "...", "severity": "high/medium/low", '
    '"action": "one-sentence recommendation"}\n\n' + test_text
)

# 3. Few-shot with examples
fs = prompt_template(
    task_type='classify',
    input_text=test_text,
    examples=[
        {'input': 'I was charged twice for my ad campaign', 'output': '{"category": "billing", "severity": "high", "action": "Initiate refund investigation"}'},
        {'input': 'Seller dashboard crashes on login', 'output': '{"category": "technical", "severity": "medium", "action": "Escalate to engineering team"}'},
        {'input': 'Inbound shipment marked lost at the fulfillment center', 'output': '{"category": "delivery", "severity": "high", "action": "Open carrier claim and reconcile inventory"}'},
    ],
    output_format='JSON with category, severity, action fields'
)

print('=' * 70)
print('FINAL COMPARISON: Same text, three approaches')
print('=' * 70)
print()
print('Text:', test_text)
print()
print('--- 1. NO TECHNIQUE (raw) ---')
print(raw[:200])
print()
print('--- 2. ZERO-SHOT (format control) ---')
print(zs)
print()
print('--- 3. FEW-SHOT (examples + format) ---')
print(json.dumps(fs, indent=2) if isinstance(fs, dict) else fs)
print()
print('=' * 70)

<div class='where-box'>
<strong>One subtlety in the final run:</strong> the few-shot version returned <code>"category": "account/payouts"</code> &mdash; a label that is not in our four-category list, because the examples showed the JSON <em>shape</em> but never stated a closed category list. The zero-shot version, which spelled out the allowed categories, stayed inside them. The production answer is to <strong>combine</strong> the techniques: an explicit closed list of labels (zero-shot constraint) <em>plus</em> examples for the ambiguous boundaries (few-shot). Constraints control the vocabulary; examples control the judgment.
</div>

<div class='takeaway'>
<strong>Key Takeaways from E02</strong>

<ol>
<li><strong>Prompt engineering is the highest-leverage GenAI skill.</strong>
The same model produces wildly different outputs based on how you ask.
Better prompts beat bigger models in most real-world tasks.</li>

<li><strong>Zero-shot works when the task is well-defined.</strong>
For classification, extraction, summarization, and translation &mdash;
clear instructions + format specs are often all you need.</li>

<li><strong>The zero-shot formula:</strong> Role + Task + Constraints + Format.
Specify WHO the model is, WHAT to do, the RULES to follow, and HOW to reply.</li>

<li><strong>System prompts set the stage.</strong> They define the model's
persona and guardrails. Same question + different system prompt = completely
different response.</li>

<li><strong>Few-shot teaches by example.</strong> Use it when categories are
domain-specific, boundaries are ambiguous, or output style must be consistent
&mdash; like our support team's "greyed-out payment option = billing" convention,
or the house-tone catalog copy.</li>

<li><strong>Few-shot can backfire.</strong> Examples can pattern-lock the model
on creative tasks. Use diverse examples, or skip them entirely for open-ended work.</li>

<li><strong>Always validate structured output.</strong> Use <code>json.loads()</code>
to confirm the model produced parseable JSON. Build retry logic for production.</li>

<li><strong>The reusable template</strong> (<code>prompt_template()</code>) combines
all techniques: task instruction, optional examples, format control, and system prompts.
This is your foundation for the rest of the week.</li>
</ol>
</div>

<div class='concept-box'>
<strong>&#128218; Glossary</strong><br>
<ul>
<li><strong>Zero-shot prompting</strong> &mdash; asking the model to do a task with instructions only, no examples.</li>
<li><strong>Few-shot prompting</strong> &mdash; including 2&ndash;5 input&rarr;output examples in the prompt to teach a pattern.</li>
<li><strong>System prompt</strong> &mdash; the hidden "job description" channel that sets persona, rules, and format.</li>
<li><strong>Format control</strong> &mdash; specifying the exact output shape (e.g. JSON schema, single word) in the prompt.</li>
<li><strong>Pattern lock</strong> &mdash; when few-shot examples bias the model into a narrow surface pattern, hurting creativity.</li>
<li><strong>Machine-parseable</strong> &mdash; output that code can consume directly (<code>json.loads()</code>, exact labels) without a human in the loop.</li>
</ul>
</div>

<hr class='section-divider'>

### &#128279; The gap we leave &mdash; Up Next: E03, Chain-of-Thought and Tree-of-Thought

<div class='concept-box'>
Few-shot got us a long way today: the <strong>format</strong> is locked, the
<strong>tone</strong> is consistent, and the classifier respects our team's
category boundaries. But watch what happens when the task requires
<strong>multi-step reasoning</strong> &mdash; the model still leaps straight
to an answer:<br><br>

&bull; <em>"A fulfillment center has 47 pallets. 23 ship out, then 15 more arrive. How many now &mdash; and do we breach dock capacity?"</em><br>
&bull; <em>"Should this seller's payout hold be released, given these 5 policy conditions?"</em><br>
&bull; <em>"Is this incident a SEV1 or SEV2, given the runbook's escalation criteria?"</em><br><br>

Zero-shot and few-shot control <em>how the model answers</em>; they do nothing
about <em>how it thinks</em> before answering. On multi-step problems, an
answer-first leap is where wrong-but-confident outputs come from.<br><br>

That is exactly the gap E03 closes: <strong>Chain-of-Thought</strong> and
<strong>Tree-of-Thought</strong> prompting make the model <em>show its work</em>
&mdash; and you will see reasoning dramatically improve accuracy on complex problems.
</div>